In [ ]:
import google.generativeai as genai 
import speech_recognition as sr
import pyttsx3
import os
import re
import threading
from pynput import keyboard
from dotenv import load_dotenv
import time

# Load API Key
load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("❌ API Key not found! Please set GEMINI_API_KEY in the .env file.")

# Configure Gemini AI
genai.configure(api_key=gemini_api_key)
model = genai.GenerativeModel("gemini-1.5-pro-latest")

# Initialize TTS Engine
engine = pyttsx3.init()
engine.setProperty("rate", 180)
engine.setProperty("volume", 1.0)
engine.setProperty("voice", engine.getProperty("voices")[0].id)

# Global Flags
stop_speech = False
listening = True  # Flag to control when the system listens
speech_lock = threading.Lock()  # Prevents multiple speech threads

def generate_ai_response(prompt, retries=3):
    """Generates AI response efficiently with retry mechanism."""
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            if response and hasattr(response, "text"):
                return response.text.strip()
            elif isinstance(response, str):
                return response.strip()
            return "I'm sorry, I couldn't process that."
        except Exception as e:
            print(f"⚠️ AI Error (Attempt {attempt+1}/{retries}): {str(e)}")
            time.sleep(0.5)  # Reduced delay for efficiency
    return "❌ Unable to process request after multiple attempts."

def on_key_press(key):
    """Handles key press to stop speech."""
    global stop_speech
    if hasattr(key, 'char') and key.char == 's':  # Press 's' to stop speech
        stop_speech = True
        engine.stop()
        print("⏹️ Speech Stopped!")

listener = keyboard.Listener(on_press=on_key_press)
listener.start()

def clean_text(text):
    """Removes unnecessary symbols for better speech output."""
    return re.sub(r'[\*\•\-_]', '', text).strip()

def speak(text):
    """Speaks the given text while ensuring no conflicts."""
    global stop_speech
    with speech_lock:  # Prevent multiple calls from conflicting
        stop_speech = False
        cleaned_text = clean_text(text)
        print(f"🤖 AI: {cleaned_text}")
        if not engine._inLoop:
            try:
                engine.say(cleaned_text)
                engine.runAndWait()
            except RuntimeError:
                print("⚠️ Speech engine error! Retrying...")
                engine.stop()
                engine.say(cleaned_text)
                engine.runAndWait()

def listen():
    """Captures voice input with improved handling."""
    global listening
    if not listening:
        return None
    
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("\n🎤 Listening...")
        recognizer.adjust_for_ambient_noise(source, duration=0.3)  # Faster adjustment
        try:
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
            text = recognizer.recognize_google(audio).lower()
            print(f"🗣️ You: {text}")  # Show recognized text immediately
            return text
        except sr.WaitTimeoutError:
            print("⏳ No speech detected.")
        except sr.UnknownValueError:
            print("🤷 Sorry, I didn't understand.")
        except sr.RequestError:
            print("⚠️ Connection issue. Please check your internet.")
    return None

# Start AI Chat
welcome_message = "Hello! I am your Health AI Assistant. You can ask me anything. Say 'exit' to stop."
print("\n🤖 AI:", welcome_message)
speak(welcome_message)

while listening:
    user_input = listen()
    if user_input:
        if any(word in user_input for word in ["exit", "bye", "stop", "quit"]):
            speak("Goodbye! Stay safe and take care! 😊")
            listening = False  # Stop listening
            break
        
        # Generate and Speak AI Response
        prompt = (f"You are an AI assistant with capabilities similar to Alexa and Siri. "
                  f"Provide a natural, engaging, and informative response to: '{user_input}'. "
                  "Ensure clarity, brevity, and completeness in your answer.")
        response = generate_ai_response(prompt)
        speak(response)
        time.sleep(0.5)  # Brief pause to enhance response timing


In [ ]:
pip install pyaudio